## 举例1：基于内存存储

临时的内存存储，进程结束则状态丢失，生产环境不可接受。

使用PostgreSQL作为持久化器，进程结束后状态会保存在数据库中，生产环境可以接受。


示例代码中可以发现，每次Cell执行的输出几乎相同，而我们并没有更改thread_id ，之所以看不到上次运行的状态是因为每次运行创建新的Saver()，历史State被丢弃了。

也就是说消息列表没有跨进程（代码执行）传递，每次运行都是独立的。

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

In [ ]:

# 连续运行两次，两次的消息列表几乎完全一样，单次运行内构成一个临时上下文，进程切换后，上下文丢失。

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

print("=" * 30, "-> 第一次调用 <-", "=" * 30)
response1 = agent.invoke(
    {"messages": [HumanMessage("你好，我是谁？")]},
    config=config
)

print(f"第一次调用后消息长度：{len(response1['messages'])}")

for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, "-> 第二次调用 <-", "=" * 30)
response2 = agent.invoke(
    {"messages": [HumanMessage("我是老王")]},
    config=config
)

print(f"第二次调用后消息长度：{len(response2['messages'])}")
for msg in response2["messages"]:
    msg.pretty_print()



print("=" * 30, "-> 第三次调用 <-", "=" * 30)
response3 = agent.invoke(
    {"messages": [HumanMessage("你好，我是谁？")]},
    config
)
print(f"第三次调用后消息长度：{len(response3['messages'])}")



for msg in response3["messages"]:
    msg.pretty_print()

============================== -> 第一次调用 <- ==============================
第一次调用后消息长度：2
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

你好！我不知道你的真实身份，除非你告诉我更多信息。  
如果你愿意，我可以根据你提供的线索帮你判断，比如：

- 你的名字或昵称
- 你现在在做什么
- 你和我之间的上下文

如果这是在开玩笑，那答案可能是：**你就是你自己**。
============================== -> 第二次调用 <- ==============================
第二次调用后消息长度：4
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

你好！我不知道你的真实身份，除非你告诉我更多信息。  
如果你愿意，我可以根据你提供的线索帮你判断，比如：

- 你的名字或昵称
- 你现在在做什么
- 你和我之间的上下文

如果这是在开玩笑，那答案可能是：**你就是你自己**。
================================ Human Message =================================

我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。  
今天想聊点什么，或者我能帮你做什么？
============================== -> 第三次调用 <- ===

## 举例2：基于PostgreSQL持久化

通过示例代码可知，上下文记忆被成功持久化到了PostgreSQL数据库中。

在后续的调用中，能够成功地从数据库中读取到之前的上下文记忆。

由此可以得出结论：即便重新创建Saver()，只要thread_id一致，历史状态就可以和当前调用串联起来。

In [4]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.postgres import PostgresSaver
import os
from dotenv import load_dotenv
load_dotenv(override=True)

DB_URL = os.getenv("DATABASE_URL")
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # 初始化数据库
    checkpointer.setup()

    agent = create_agent(
        model=model,
        checkpointer=checkpointer
    )

    config = {"configurable": {"thread_id": "3"}}

    print("=" * 30, "-> 第一次调用 <-", "=" * 30)
    response1 = agent.invoke(
        {"messages": [HumanMessage("你好，我是谁啊？")]},
        config
    )
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, "-> 第二次调用 <-", "=" * 30)
    response2 = agent.invoke(
        {"messages": [HumanMessage("我是老王，我喜欢吃KFC，爱好爬山。")]},
        config
    )
    for msg in response2["messages"]:
        msg.pretty_print()

    print("=" * 30, "-> 第三次调用 <-", "=" * 30)
    response3 = agent.invoke(
        {"messages": [HumanMessage("你好，我是谁？？")]},
        config
    )

for msg in response3["messages"]:
    msg.pretty_print()

print(f"第1次Cell运行后总消息数： {len(response3['messages'])}") 

============================== -> 第一次调用 <- ==============================
================================ Human Message =================================

你好，我是谁啊？
================================== Ai Message ==================================

你好！我现在还不知道你具体是谁，因为我不会自动看到你的身份信息。

如果你愿意，我可以根据你告诉我的内容来“认识你”：
- 你的名字或昵称
- 你是做什么的
- 你想让我怎么称呼你

如果你是想问“在这次对话里我记得你吗”，那答案是：**我只能根据当前对话里你告诉我的内容来判断**。
============================== -> 第二次调用 <- ==============================
================================ Human Message =================================

你好，我是谁啊？
================================== Ai Message ==================================

你好！我现在还不知道你具体是谁，因为我不会自动看到你的身份信息。

如果你愿意，我可以根据你告诉我的内容来“认识你”：
- 你的名字或昵称
- 你是做什么的
- 你想让我怎么称呼你

如果你是想问“在这次对话里我记得你吗”，那答案是：**我只能根据当前对话里你告诉我的内容来判断**。
================================ Human Message =================================

我是老王，我喜欢吃KFC，爱好爬山。
================================== Ai Message ==================================

你好，老王！很高兴认识你。

我记住你现在告诉我的信息了：
- 你叫 **

In [5]:
from rich import print as rich_print
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # 初始化数据库
    checkpointer.setup()

    agent2 = create_agent(
        model=model,
        checkpointer=checkpointer
    )

    config = {"configurable": {"thread_id": "3"}}
    response4 = agent2.invoke(
        {"messages": [HumanMessage("你好，你还记得我吗？")]},
        config=config
    )

print(f"第2次Cell运行后总消息数：: {len(response4['messages'])}")
rich_print(response4)

第2次Cell运行后总消息数：: 8


{
    'messages': [
        HumanMessage(
            content='你好，我是谁啊？',
            additional_kwargs={},
            response_metadata={},
            id='65fed7e1-e408-4d1c-9e9f-bc5a4e618b80'
        ),
        AIMessage(
            content='你好！我现在还不知道你具体是谁，因为我不会自动看到你的身份信息。\n\n如果你愿意，我可以根据你告
诉我的内容来“认识你”：\n- 你的名字或昵称\n- 你是做什么的\n- 
你想让我怎么称呼你\n\n如果你是想问“在这次对话里我记得你吗”，那答案是：**我只能根据当前对话里你告诉我的内容来判断**
。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 102,
                    'prompt_tokens': 11,
                    'total_tokens': 113,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00046725,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00046725,
                        'upstream_inference_prompt_cost': 8.25e-06,
                        'upstream_inference_completions_cost': 0.000459
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1785726516-irWEoDKPeNDvJio0GJzB',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019fc598-30db-7951-a166-a92206d1cef6-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 11,
                'output_tokens': 102,
                'total_tokens': 113,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        HumanMessage(
            content='我是老王，我喜欢吃KFC，爱好爬山。',
            additional_kwargs={},
            response_metadata={},
            id='1edb28f5-48da-47ae-89b6-6077c372998a'
        ),
        AIMessage(
            content='你好，老王！很高兴认识你。\n\n我记住你现在告诉我的信息了：\n- 你叫 **老王**\n- 你喜欢吃 
**KFC**\n- 你的爱好是 **爬山**\n\n以后你可以直接这样跟我说话，比如：\n- “老王来啦”\n- “我想聊聊爬山”\n- 
“给我推荐点KFC吃什么”\n\n如果你愿意，我也可以顺便帮你做一个“老王简介”。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 110,
                    'prompt_tokens': 134,
                    'total_tokens': 244,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.0005955,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0005955,
                        'upstream_inference_prompt_cost': 0.0001005,
                        'upstream_inference_completions_cost': 0.000495
                    }
                },
                'model_provider': 'openai',
                'model_name': '



**总结：**
1. `InMemorySaver()`将状态持久化到内存，进程结束或重建`Saver()`则历史状态丢失
2. 基于外部存储介质（如PostgreSQL）的持久化器，其存储的状态不会随进程终止而丢失，只要不显式删除历史状态，即可通过`thread_id`加载历史状态。